In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
LAND = f"/Volumes/{CATALOG}/bronze/landing"

# a separate volume to hold Auto Loader's memory (checkpoint and schema)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.checkpoints")
CHK = f"/Volumes/{CATALOG}/bronze/checkpoints/orders_autoloader"

stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "orders_*.csv")          # only order files, not products or weather
    .option("cloudFiles.schemaLocation", f"{CHK}/schema")
    .load(LAND)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))

q = (stream.writeStream
     .option("checkpointLocation", f"{CHK}/ckpt")
     .trigger(availableNow=True)
     .toTable(f"{CATALOG}.bronze.orders_stream"))
q.awaitTermination()

print("rows in bronze.orders_stream:", spark.table(f"{CATALOG}.bronze.orders_stream").count())

rows in bronze.orders_stream: 16163


In [0]:
display(spark.table("workspace.bronze.orders_stream")
        .groupBy("_source_file")
        .agg(F.count("*").alias("rows"), F.max("_ingested_at").alias("ingested_at"))
        .orderBy(F.desc("ingested_at"))
        .limit(3))

_source_file,rows,ingested_at
/Volumes/workspace/bronze/landing/orders_2026-11-01.csv,505,2026-11-04T09:01:47.647Z
/Volumes/workspace/bronze/landing/orders_2026-10-09.csv,505,2026-11-04T07:55:52.178Z
/Volumes/workspace/bronze/landing/orders_2026-10-10.csv,505,2026-11-04T07:55:52.178Z


In [0]:
b = spark.table("workspace.bronze.orders").count()
s = spark.table("workspace.bronze.orders_stream").count()
print("batch bronze:", b, "| auto loader bronze:", s)

batch bronze: 15658 | auto loader bronze: 16163
